# Plots scatter random forest performance

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import dask.dataframe as dd
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# Directories
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'
dir03 = '../paper_ecosystem_carbon/output/03_rf_predict/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'

---

### Pre-processing

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='interactive',
                       cores=24,
                       memory='100 GB',
                       account='bm0891', 
                       interface="ib0",
                       walltime='02:00:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=10)

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)
client

In [ ]:
# Get database
dd_base = dd.read_parquet(f'{dir02}dd_base_r001.parquet') \
    .repartition(partition_size='1000 MiB') \
    .reset_index(True) \
    .persist()

In [ ]:
# Dictionary with ecosystem types and original names
dict_etype = dict(
    trop_ef = ['Tropical/subtropical evergreen forest', 
               'Tropical/subtropical\nevergreen forest'], 
    trop_df = ['Tropical/subtropical deciduous forest', 
               'Tropical/subtropical\ndeciduous forest'],
    temp_ebf = ['Temperate/boreal evergreen broadleaf forest', 
                'Temperate/boreal\nevergreen broadleaf forest'],
    temp_dbf = ['Temperate/boreal deciduous broadleaf forest', 
                'Temperate/boreal\ndeciduous broadleaf forest'],
    temp_enf = ['Temperate/boreal evergreen needleleaf forest', 
                'Temperate/boreal\nevergreen needleleaf forest'],
    temp_dnf = ['Temperate/boreal deciduous needleleaf forest', 
                'Temperate/boreal\ndeciduous needleleaf forest'],
    glob_shrub = ['Shrublands', 'Shrublands'],
    glob_grass = ['Grasslands','Grasslands'],
    glob_tundr = ['Tundra', 'Tundra'],
    glob_mount = ['Montane', 'Montane'])

---

### Plot functions

In [ ]:
def prep_data(ctype, etype):

    """Prepares scatter data for one ecosystem type"""

    # Get predicted values
    dd_pred = dd.read_parquet(
        f'{dir03}files_predicted_r001/df_rfpred_{ctype}_{etype}.parquet') \
        .repartition(partition_size='1000 MiB') \
        .reset_index(True) \
        .persist()
    
    # Prepare dataframe with predicted values
    rfr_cols = [f'rfr_{i}' for i in range(1, 11)]
        
    dd_pred_sel = dd_pred \
        .assign(cpred = dd_pred[rfr_cols].mean(axis=1)) \
        [['lat', 'lon', 'cpred']]

    # Define training/testing carbon (ensure consistency with 02_05_prep_database_r001)
    if ctype == 'cveg':
        if etype in ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
                     'temp_dnf', 'glob_shrub']:
            cbase = 'cveg_biom'    
        if etype in ['glob_tundr', 'glob_mount']:
            cbase = 'cveg_tund'
        if etype in ['glob_grass', 'glob_past']:
            cbase = 'cveg_gras'
        if etype == 'glob_crop':
            cbase = 'cveg_crop'
    if ctype == 'csoil':
            cbase = 'csoil_000_100'

    # Prepare dataframe with testing values
    dd_base_sel = dd_base[((dd_base[f'test_{ctype}_{etype}'] == True))] \
        [['lat', 'lon', cbase]].rename(columns={cbase: 'cbase'})

    # Create dataframe with testing values (orig and predicted)
    df_test = dd.merge(dd_base_sel, dd_pred_sel,
                       how='left', on=['lat', 'lon']).compute()

    return df_test

In [ ]:
ctype, etype = 'cveg', 'glob_tundr'

#Get data
df_sel = prep_data(ctype, etype)

# Sample the data for plotting
n_sample=25000
if len(df_sel) > n_sample:
    df_sam = df_sel.sample(n=n_sample)
else: 
    df_sam = df_sel
    
# Convert to pandas for faster operations
x_sam, y_sam = df_sam['cbase'], df_sam['cpred']

# Calculate the point density using Gaussian Kernel Density Estimation
xy = np.vstack([x_sam, y_sam])
z = np.log(stats.gaussian_kde(xy)(xy))

# Perform linear regression
reg = stats.linregress(x_sam, y_sam)
m = reg.slope
b = reg.intercept

# Calculate metrics on the full dataset (using Dask for larger data)
x_full, y_full = df_sel['cbase'], df_sel['cpred']
metrics_dict = {
    'r2': metrics.r2_score(x_full, y_full),
    'rmse': metrics.root_mean_squared_error(x_full, y_full),
    'mae': metrics.mean_absolute_error(x_full, y_full),
}


In [ ]:
def plot_sub(ax, ctype, etype):

    """Plots predicted vs. testing data for one ecosystem type"""
    
    #Get data
    df_sel = prep_data(ctype, etype)

    # Sample the data for plotting
    n_sample=25000
    if len(df_sel) > n_sample:
        df_sam = df_sel.sample(n=n_sample)
    else: 
        df_sam = df_sel
        
    # Convert to pandas for faster operations
    x_sam, y_sam = df_sam['cbase'], df_sam['cpred']

    # Calculate the point density using Gaussian Kernel Density Estimation
    xy = np.vstack([x_sam, y_sam])
    z = np.log(stats.gaussian_kde(xy)(xy))

    # Perform linear regression
    reg = stats.linregress(x_sam, y_sam)
    m = reg.slope
    b = reg.intercept

    # Calculate metrics on the full dataset (using Dask for larger data)
    x_full, y_full = df_sel['cbase'], df_sel['cpred']
    metrics_dict = {
        'r2': metrics.r2_score(x_full, y_full),
        'rmse': metrics.root_mean_squared_error(x_full, y_full),
        'mae': metrics.mean_absolute_error(x_full, y_full),
    }

    # Plotting
    ax.scatter(x=x_sam, y=y_sam, c=z, s=1, rasterized=True)
    # 1:1 line
    ax.axline((0, 0), (1, 1), c='#636363', linestyle='--')
    # linear regression
    ax.axline(xy1=(0, b), slope=m, c='#000000')

    # Display metrics in the plot
    metric_text = (
        f'R2: {metrics_dict['r2']:.2f}\n'
        f'RMSE: {metrics_dict['rmse']:.2f}\n'
        f'MAE: {metrics_dict['mae']:.2f}')
    
    ax.text(0, y_sam.max(), va='top', 
            bbox=dict(facecolor='#ffffff', alpha=0.5, edgecolor='none'), 
            s=metric_text)

    # Set titles and labels
    ax.set_title(dict_etype[etype][1])
    ax.set_xlabel('Test data (tC ha$^{-1}$)')
    ax.set_ylabel('Prediction (tC ha$^{-1}$)')

---

### Plots

In [ ]:
def plot_main(ctype):
    fig, axs = plt.subplots(figsize=(10, 10), ncols=3, nrows=4, dpi=300)
    axs = axs.ravel()

    # Loop through ecosystem types and plot scatter
    for i in range(0, len(list(dict_etype.keys()))):
        etype = list(dict_etype.keys())[i]
        plot_sub(axs[i], ctype, etype)

    # Layout
    axs[10].axis('off')
    axs[11].axis('off')
    
    plt.tight_layout()

    # Export
    if ctype == 'cveg':
        fstr_out = f'fig_s02_cveg_scatter_cspatial'
    if ctype == 'csoil':
        fstr_out = f'fig_s04_csoil_scatter_cspatial'

    plt.savefig(f'{dir08}pdf/{fstr_out}.pdf', dpi=300, 
            bbox_inches='tight', pad_inches=0.1); 
    plt.savefig(f'{dir08}png/{fstr_out}.png', dpi=300, 
            bbox_inches='tight', pad_inches=0.1);

In [ ]:
plot_main('cveg')

In [ ]:
plot_main('csoil')

---

### Get p-values

In [ ]:
def get_p_values():
    rows = []
    
    for ctype in ['cveg', 'csoil']:
        for etype in dict_etype.keys():
            df_sel = prep_data(ctype, etype)
    
            x, y = df_sel['cbase'], df_sel['cpred']
    
            if hasattr(x, 'compute'):
                x = x.compute()
            if hasattr(y, 'compute'):
                y = y.compute()
    
            df_xy = pd.DataFrame({'x': x, 'y': y}).dropna()
            
            reg = stats.linregress(df_xy['x'], df_xy['y'])    
            p_value = reg.pvalue
    
            rows.append({
                'ctype': ctype,
                'etype': etype,
                'p_value': f'{p_value:.5e}',
            })
    
    return pd.DataFrame(rows)

get_p_values()

---

In [ ]:
cluster.close()